# Prévision de la demande et optimisation des stocks par l'IA
## Notebook : préparation, analyse exploratoire et premiers prototypes

**Dataset Kaggle** : *Retail Store Inventory Forecasting Dataset* (fichier `retail_store_inventory.csv`, données de ventes et de stock par magasin et par produit, au jour le jour).

**Plan**
1. Chargement et premier diagnostic
2. Nettoyage, dédoublonnage, valeurs manquantes, mise en qualité
3. Transformation et création de variables
4. Analyse statistique
5. Analyse exploratoire : tendances, saisonnalité, corrélations
6. Détection d'anomalies
7. Premiers indicateurs (KPI stock et demande)
8. Prototypes : prévision de la demande et recommandation de quantité à commander
9. Export des résultats

## 0. Configuration

In [ ]:
import glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# --- Paramètres du projet (à adapter) ---
KAGGLE_DATASET = "anirudhchauhan/retail-store-inventory-forecasting-dataset"  # à vérifier sur Kaggle
FICHIER_LOCAL  = "retail_store_inventory.csv"
LEAD_TIME_JOURS = 7      # délai de livraison supposé (absent du dataset)
JOURS_REVUE     = 7      # fréquence de réapprovisionnement
Z_SERVICE       = 1.65   # niveau de service d'environ 95 %
JOURS_TEST      = 60     # derniers jours réservés au test des modèles
SEED            = 42

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Chargement et premier diagnostic

In [ ]:
def trouver_csv():
    """Cherche le fichier : notebook Kaggle, dossier local, puis téléchargement avec kagglehub."""
    for p in glob.glob("/kaggle/input/**/*.csv", recursive=True):
        if "inventory" in p.lower():
            return p
    for p in [FICHIER_LOCAL, f"data/{FICHIER_LOCAL}"]:
        if Path(p).exists():
            return p
    try:
        import kagglehub
        dossier = kagglehub.dataset_download(KAGGLE_DATASET)
        csvs = glob.glob(f"{dossier}/**/*.csv", recursive=True)
        if csvs:
            return csvs[0]
    except Exception as e:
        print("Téléchargement automatique impossible :", e)
    raise FileNotFoundError(
        "Fichier introuvable. Téléchargez retail_store_inventory.csv sur Kaggle "
        "et placez-le à côté du notebook."
    )

chemin = trouver_csv()
print("Fichier utilisé :", chemin)
brut = pd.read_csv(chemin)
print("Dimensions :", brut.shape)
brut.head()

In [ ]:
brut.info()
display(brut.describe(include="all").T)

## 2. Nettoyage et mise en qualité

Étapes : normalisation des noms de colonnes, typage, nettoyage des textes, valeurs impossibles, dédoublonnage, valeurs manquantes (avec indicateurs de traçabilité), valeurs extrêmes. Un rapport avant / après est produit à la fin.

In [ ]:
def rapport_qualite(d, nom):
    n = len(d)
    manquants = int(d.isna().sum().sum())
    return {
        "jeu": nom, "lignes": n, "colonnes": d.shape[1],
        "valeurs_manquantes": manquants,
        "pct_manquantes": round(100 * manquants / (n * d.shape[1]), 3) if n else 0,
        "doublons_exacts": int(d.duplicated().sum()),
    }

rapports = [rapport_qualite(brut, "avant nettoyage")]
journal = []   # traçabilité des opérations

df = brut.copy()

# 2.1 Noms de colonnes en snake_case
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^0-9a-z]+", "_", regex=True).str.strip("_"))
print("Colonnes :", list(df.columns))

obligatoires = ["date", "store_id", "product_id", "units_sold"]
manquantes = [c for c in obligatoires if c not in df.columns]
assert not manquantes, f"Colonnes obligatoires absentes : {manquantes}"

In [ ]:
# 2.2 Typage et nettoyage des textes
df["date"] = pd.to_datetime(df["date"], errors="coerce")
n_date_invalide = int(df["date"].isna().sum())
df = df.dropna(subset=["date"])
journal.append(f"Lignes supprimées (date invalide) : {n_date_invalide}")

for c in ["store_id", "product_id"]:
    df[c] = df[c].astype(str).str.strip().str.upper()

cat_cols = [c for c in ["category", "region", "weather_condition", "seasonality"] if c in df.columns]
for c in cat_cols:
    df[c] = (df[c].astype("string").str.strip().str.title()
             .replace({"": pd.NA, "Nan": pd.NA, "None": pd.NA, "Null": pd.NA}))

num_cols = [c for c in ["inventory_level", "units_sold", "units_ordered", "demand_forecast",
                        "price", "discount", "competitor_pricing"] if c in df.columns]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if "holiday_promotion" in df.columns:
    df["holiday_promotion"] = pd.to_numeric(df["holiday_promotion"], errors="coerce")

In [ ]:
# 2.3 Valeurs impossibles -> converties en valeurs manquantes (puis traitées)
n_neg = 0
for c in [c for c in ["inventory_level", "units_sold", "units_ordered", "demand_forecast",
                      "price", "competitor_pricing"] if c in df.columns]:
    masque = df[c] < 0
    n_neg += int(masque.sum())
    df.loc[masque, c] = np.nan
if "discount" in df.columns:
    masque = (df["discount"] < 0) | (df["discount"] > 100)
    n_neg += int(masque.sum())
    df.loc[masque, "discount"] = np.nan
journal.append(f"Valeurs impossibles (négatives ou hors bornes) mises à NaN : {n_neg}")

# 2.4 Dédoublonnage
n0 = len(df)
df = df.drop_duplicates()
journal.append(f"Doublons exacts supprimés : {n0 - len(df)}")

cle = ["date", "store_id", "product_id"]
n1 = len(df)
df = df.sort_values(cle).drop_duplicates(subset=cle, keep="last")
journal.append(f"Doublons sur la clé (date, magasin, produit) supprimés : {n1 - len(df)}")
print("\n".join(journal))

In [ ]:
# 2.5 Valeurs manquantes : indicateurs de traçabilité puis imputation
print("Valeurs manquantes avant imputation :")
display(df.isna().sum()[lambda s: s > 0].to_frame("manquantes"))

for c in num_cols + ["holiday_promotion"]:
    if c in df.columns and df[c].isna().any():
        df[f"{c}_was_missing"] = df[c].isna().astype(int)

df = df.sort_values(["store_id", "product_id", "date"]).reset_index(drop=True)
grp = ["store_id", "product_id"]

for c in num_cols:
    if df[c].isna().any():
        # médiane de la série (magasin x produit), puis médiane globale en secours
        df[c] = df[c].fillna(df.groupby(grp)[c].transform("median"))
        df[c] = df[c].fillna(df[c].median())

if "holiday_promotion" in df.columns:
    df["holiday_promotion"] = df["holiday_promotion"].fillna(0).clip(0, 1).astype(int)

for c in cat_cols:
    if df[c].isna().any():
        # catégorie du produit : valeur la plus fréquente ; sinon "Inconnu"
        mode = df.groupby("product_id")[c].transform(lambda s: s.mode().iat[0] if s.notna().any() else pd.NA)
        df[c] = df[c].fillna(mode).fillna("Inconnu")

print("Valeurs manquantes restantes :", int(df.isna().sum().sum()))

In [ ]:
# 2.6 Valeurs extrêmes : marquage (IQR large, par produit) et version "winsorisée" des ventes
q1 = df.groupby("product_id")["units_sold"].transform(lambda s: s.quantile(0.25))
q3 = df.groupby("product_id")["units_sold"].transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
borne_basse, borne_haute = q1 - 3 * iqr, q3 + 3 * iqr
df["outlier_units"] = ((df["units_sold"] < borne_basse) | (df["units_sold"] > borne_haute)).astype(int)
df["units_sold_winsor"] = df["units_sold"].clip(lower=borne_basse.clip(lower=0), upper=borne_haute)
journal.append(f"Ventes extrêmes marquées (IQR x3 par produit) : {int(df['outlier_units'].sum())}")

# 2.7 Contrôles de cohérence
if "inventory_level" in df.columns:
    df["ventes_sup_stock"] = (df["units_sold"] > df["inventory_level"]).astype(int)
    journal.append(f"Lignes où les ventes dépassent le stock affiché : {int(df['ventes_sup_stock'].sum())}")

# Continuité des dates par série
cont = df.groupby(grp)["date"].agg(["min", "max", "nunique"])
cont["attendu"] = (cont["max"] - cont["min"]).dt.days + 1
cont["jours_manquants"] = cont["attendu"] - cont["nunique"]
journal.append(f"Jours manquants dans les séries : {int(cont['jours_manquants'].sum())}")

rapports.append(rapport_qualite(df, "après nettoyage"))
print("\n".join(journal))
display(pd.DataFrame(rapports).set_index("jeu"))

## 3. Transformation et création de variables

Variables calendaires, retards (*lags*), moyennes glissantes, prix net, écart au prix concurrent et couverture de stock. Les retards et moyennes glissantes sont calculés **par série** (magasin x produit) et décalés pour ne pas utiliser le jour à prédire.

In [ ]:
df = df.sort_values(["store_id", "product_id", "date"]).reset_index(drop=True)

# Calendrier
df["annee"] = df["date"].dt.year
df["mois"] = df["date"].dt.month
df["semaine"] = df["date"].dt.isocalendar().week.astype(int)
df["jour_semaine"] = df["date"].dt.dayofweek
df["weekend"] = (df["jour_semaine"] >= 5).astype(int)

# Retards et moyennes glissantes (par série, sans fuite d'information)
g = df.groupby(grp)["units_sold"]
for lag in (1, 7, 14):
    df[f"lag_{lag}"] = g.shift(lag)
df["roll_mean_7"] = g.transform(lambda s: s.shift(1).rolling(7).mean())
df["roll_mean_28"] = g.transform(lambda s: s.shift(1).rolling(28).mean())
df["roll_std_28"] = g.transform(lambda s: s.shift(1).rolling(28).std())
if "inventory_level" in df.columns:
    df["inventory_lag_1"] = df.groupby(grp)["inventory_level"].shift(1)

# Prix
if {"price", "discount"} <= set(df.columns):
    df["price_net"] = df["price"] * (1 - df["discount"] / 100)
if {"price_net", "competitor_pricing"} <= set(df.columns):
    df["ecart_prix_concurrent"] = df["price_net"] / df["competitor_pricing"].replace(0, np.nan) - 1

# Stock
if "inventory_level" in df.columns:
    df["couverture_jours"] = df["inventory_level"] / df["roll_mean_28"].replace(0, np.nan)
    df["tension_stock"] = (df["units_sold"] >= df["inventory_level"]).astype(int)  # hypothèse : ventes >= stock

print(df.shape)
df.head()

## 4. Analyse statistique

In [ ]:
num_stats = [c for c in ["units_sold", "inventory_level", "units_ordered", "demand_forecast",
                         "price", "discount", "competitor_pricing"] if c in df.columns]
resume = df[num_stats].describe().T
resume["asymetrie"] = df[num_stats].skew()
resume["aplatissement"] = df[num_stats].kurt()
resume["cv"] = resume["std"] / resume["mean"]
display(resume)

In [ ]:
# Normalité des ventes (échantillon) : test de D'Agostino
echantillon = df["units_sold"].sample(min(5000, len(df)), random_state=SEED)
stat, p = stats.normaltest(echantillon)
print(f"Normalité des ventes : statistique = {stat:.1f}, p = {p:.4g} ->",
      "distribution non normale" if p < 0.05 else "compatible avec une loi normale")

# Effet de la catégorie (Kruskal-Wallis)
if "category" in df.columns:
    groupes = [s["units_sold"].values for _, s in df.groupby("category")]
    h, p = stats.kruskal(*groupes)
    print(f"Ventes selon la catégorie : H = {h:.1f}, p = {p:.4g} ->",
          "différences significatives" if p < 0.05 else "pas de différence significative")

# Effet des promotions / jours fériés (Mann-Whitney)
if "holiday_promotion" in df.columns and df["holiday_promotion"].nunique() > 1:
    a = df.loc[df["holiday_promotion"] == 1, "units_sold"]
    b = df.loc[df["holiday_promotion"] == 0, "units_sold"]
    u, p = stats.mannwhitneyu(a, b)
    print(f"Promotion/férié : moyenne {a.mean():.1f} contre {b.mean():.1f} "
          f"({100 * (a.mean() / b.mean() - 1):+.1f} %), p = {p:.4g}")

# Remise, prix et ventes (corrélation de Spearman)
for c in [c for c in ["discount", "price_net", "ecart_prix_concurrent"] if c in df.columns]:
    r, p = stats.spearmanr(df[c], df["units_sold"])
    print(f"Spearman units_sold ~ {c} : rho = {r:+.3f}, p = {p:.4g}")

## 5. Analyse exploratoire : tendances, saisonnalité, corrélations

In [ ]:
# 5.1 Ventes totales par jour, moyennes mobiles et tendance
jour = df.groupby("date")["units_sold"].sum().rename("ventes").to_frame()
jour["ma7"] = jour["ventes"].rolling(7).mean()
jour["ma28"] = jour["ventes"].rolling(28).mean()

ax = jour[["ventes", "ma7", "ma28"]].plot(alpha=0.9, lw=1.2)
ax.set_title("Ventes totales par jour"); ax.set_ylabel("Unités"); plt.show()

x = np.arange(len(jour))
reg = stats.linregress(x, jour["ventes"].values)
variation = 100 * reg.slope * 365 / jour["ventes"].mean()
print(f"Tendance linéaire : {reg.slope:+.2f} unités/jour par jour (p = {reg.pvalue:.3g}) "
      f"soit environ {variation:+.1f} % par an")

In [ ]:
# 5.2 Décomposition simple : tendance (moyenne mobile centrée), saisonnalité hebdomadaire, résidu
serie = jour["ventes"]
tendance = serie.rolling(7, center=True).mean()
detrend = serie - tendance
saison = detrend.groupby(serie.index.dayofweek).transform("mean")
residu = serie - tendance - saison

fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for ax, s, t in zip(axes, [serie, tendance, saison, residu],
                    ["Série observée", "Tendance", "Saisonnalité hebdomadaire", "Résidu"]):
    ax.plot(s, lw=1); ax.set_title(t, loc="left", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# 5.3 Saisonnalité : mois et jour de semaine
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby("mois")["units_sold"].mean().plot.bar(ax=axes[0], color="#4472C4")
axes[0].set_title("Ventes moyennes par mois"); axes[0].set_xlabel("Mois")
noms = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
moy_js = df.groupby("jour_semaine")["units_sold"].mean()
axes[1].bar([noms[i] for i in moy_js.index], moy_js.values, color="#F2A900")
axes[1].set_title("Ventes moyennes par jour de la semaine")
plt.tight_layout(); plt.show()

In [ ]:
# 5.4 Segments : catégorie, région, saisonnalité, météo
segments = [c for c in ["category", "region", "seasonality", "weather_condition"] if c in df.columns]
if segments:
    fig, axes = plt.subplots(1, len(segments), figsize=(4.5 * len(segments), 4))
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, segments):
        df.groupby(c)["units_sold"].mean().sort_values().plot.barh(ax=ax, color="#4472C4")
        ax.set_title(f"Ventes moyennes par {c}")
    plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["units_sold"], bins=50, kde=True, ax=axes[0], color="#4472C4")
axes[0].set_title("Distribution des ventes quotidiennes")
if "category" in df.columns:
    sns.boxplot(data=df, x="category", y="units_sold", ax=axes[1])
    axes[1].tick_params(axis="x", rotation=30); axes[1].set_title("Ventes par catégorie")
plt.tight_layout(); plt.show()

In [ ]:
# 5.5 Corrélations (Spearman, plus robuste aux valeurs extrêmes)
cols_corr = [c for c in ["units_sold", "inventory_level", "units_ordered", "demand_forecast", "price",
                         "discount", "price_net", "competitor_pricing", "ecart_prix_concurrent",
                         "holiday_promotion", "lag_1", "lag_7", "roll_mean_7", "roll_mean_28", "mois", "jour_semaine"]
             if c in df.columns]
corr = df[cols_corr].corr(method="spearman")
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, annot_kws={"size": 7})
plt.title("Matrice de corrélation (Spearman)"); plt.show()

print("Corrélations les plus fortes avec units_sold :")
display(corr["units_sold"].drop("units_sold").sort_values(key=abs, ascending=False).head(8).to_frame("rho"))

In [ ]:
# 5.6 Promotions, remises et stock
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
if "holiday_promotion" in df.columns:
    df.groupby("holiday_promotion")["units_sold"].mean().plot.bar(ax=axes[0], color=["#999999", "#F2A900"])
    axes[0].set_title("Ventes moyennes : sans / avec promotion ou férié")
if "discount" in df.columns:
    df.groupby("discount")["units_sold"].mean().plot(marker="o", ax=axes[1])
    axes[1].set_title("Ventes moyennes selon la remise (%)")
if "inventory_level" in df.columns:
    par_produit = df.groupby("product_id")[["inventory_level", "units_sold"]].mean()
    axes[2].scatter(par_produit["inventory_level"], par_produit["units_sold"], color="#4472C4")
    axes[2].set_xlabel("Stock moyen"); axes[2].set_ylabel("Ventes moyennes")
    axes[2].set_title("Stock moyen et ventes moyennes par produit")
plt.tight_layout(); plt.show()

## 6. Détection d'anomalies

In [ ]:
# 6.1 Série quotidienne : z-score robuste (médiane / MAD) sur le résidu de la décomposition
r = residu.dropna()
mad = np.median(np.abs(r - r.median()))
z_rob = 0.6745 * (r - r.median()) / (mad if mad > 0 else 1)
anom_jours = z_rob[np.abs(z_rob) > 3.5]
print(f"{len(anom_jours)} jour(s) anormal(aux) sur {len(r)}")

plt.plot(serie, lw=1, label="Ventes")
plt.scatter(anom_jours.index, serie.loc[anom_jours.index], color="red", zorder=3, label="Anomalies")
plt.title("Anomalies sur les ventes totales quotidiennes"); plt.legend(); plt.show()
display(pd.DataFrame({"ventes": serie.loc[anom_jours.index], "z_robuste": anom_jours}).head(10))

In [ ]:
# 6.2 Lignes (magasin x produit x jour) : Isolation Forest sur plusieurs variables
from sklearn.ensemble import IsolationForest

vars_anom = [c for c in ["units_sold", "inventory_level", "units_ordered", "price_net", "discount"] if c in df.columns]
iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=SEED, n_jobs=-1)
df["anomalie_iforest"] = (iso.fit_predict(df[vars_anom]) == -1).astype(int)
df["score_anomalie"] = -iso.score_samples(df[vars_anom])

print("Lignes anormales (Isolation Forest) :", int(df["anomalie_iforest"].sum()))
print("Ventes extrêmes (IQR x3)           :", int(df["outlier_units"].sum()))
anomalies = (df[df["anomalie_iforest"] == 1]
             .sort_values("score_anomalie", ascending=False)
             [["date", "store_id", "product_id"] + vars_anom + ["score_anomalie"]])
anomalies.head(10)

## 7. Premiers indicateurs (KPI)

Indicateurs par série (magasin x produit) : demande moyenne et variabilité, couverture de stock, rotation, tension de stock, classification **ABC** (poids dans les ventes) et **XYZ** (régularité de la demande), précision de la prévision fournie, puis **stock de sécurité, point de commande et quantité à commander**.

In [ ]:
n_jours = (df["date"].max() - df["date"].min()).days + 1

kpi = df.groupby(grp).agg(
    demande_moy=("units_sold", "mean"),
    demande_std=("units_sold", "std"),
    ventes_totales=("units_sold", "sum"),
    stock_moyen=("inventory_level", "mean"),
    stock_actuel=("inventory_level", "last"),
    tension_stock_pct=("tension_stock", "mean"),
    prix_net_moyen=("price_net", "mean"),
).reset_index()

if "category" in df.columns:
    cat_map = df.groupby("product_id")["category"].agg(lambda s: s.mode().iat[0]).rename("category")
    kpi = kpi.merge(cat_map, on="product_id", how="left")

kpi["cv_demande"] = kpi["demande_std"] / kpi["demande_moy"]
kpi["couverture_jours"] = kpi["stock_moyen"] / kpi["demande_moy"]
kpi["rotation_annuelle"] = kpi["ventes_totales"] / kpi["stock_moyen"] * (365 / n_jours)
kpi["tension_stock_pct"] *= 100

# ABC (poids cumulé dans les ventes) et XYZ (terciles du coefficient de variation)
kpi = kpi.sort_values("ventes_totales", ascending=False)
part_cum = kpi["ventes_totales"].cumsum() / kpi["ventes_totales"].sum()
kpi["classe_ABC"] = np.select([part_cum <= 0.8, part_cum <= 0.95], ["A", "B"], default="C")
kpi["classe_XYZ"] = pd.qcut(kpi["cv_demande"].rank(method="first"), 3, labels=["X", "Y", "Z"])

# Précision de la prévision déjà présente dans le dataset (si disponible)
if "demand_forecast" in df.columns:
    err = df["units_sold"] - df["demand_forecast"]
    wape = err.abs().sum() / df["units_sold"].sum()
    print(f"Prévision fournie : WAPE = {100 * wape:.1f} %, biais moyen = {err.mean():+.2f} unités/jour")

kpi.head(10)

In [ ]:
# Politique de stock : stock de sécurité, point de commande, niveau cible, quantité à commander
L, R = LEAD_TIME_JOURS, JOURS_REVUE
kpi["stock_securite"] = Z_SERVICE * kpi["demande_std"] * np.sqrt(L)
kpi["point_commande"] = kpi["demande_moy"] * L + kpi["stock_securite"]
kpi["niveau_cible"] = kpi["demande_moy"] * (L + R) + kpi["stock_securite"]
kpi["qte_a_commander"] = np.ceil((kpi["niveau_cible"] - kpi["stock_actuel"]).clip(lower=0))
kpi["statut"] = np.where(kpi["stock_actuel"] < kpi["point_commande"], "Risque de rupture", "OK")

print("Répartition ABC / XYZ :")
display(pd.crosstab(kpi["classe_ABC"], kpi["classe_XYZ"]))
print("Statut des séries :", kpi["statut"].value_counts().to_dict())
kpi.sort_values("qte_a_commander", ascending=False).head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(kpi["couverture_jours"], bins=25, ax=axes[0], color="#4472C4"); axes[0].set_title("Couverture de stock (jours)")
sns.histplot(kpi["rotation_annuelle"], bins=25, ax=axes[1], color="#F2A900"); axes[1].set_title("Rotation annuelle")
kpi["statut"].value_counts().plot.bar(ax=axes[2], color=["#2E8B57", "#C0392B"]); axes[2].set_title("Statut des séries")
plt.tight_layout(); plt.show()

## 8. Prototypes

### 8.1 Prévision de la demande
Découpage **temporel** (les derniers jours servent de test). Comparaison de trois références simples (même jour la semaine précédente, moyenne glissante 7 jours, prévision fournie par le dataset) avec un modèle de *gradient boosting*.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

features_num = [c for c in ["lag_1", "lag_7", "lag_14", "roll_mean_7", "roll_mean_28", "roll_std_28",
                            "inventory_lag_1", "price_net", "discount", "competitor_pricing",
                            "ecart_prix_concurrent", "holiday_promotion", "mois", "jour_semaine", "weekend"]
                if c in df.columns]
features_cat = [c for c in ["category", "region", "weather_condition", "seasonality"] if c in df.columns]

modele_df = df.dropna(subset=["lag_14", "roll_mean_28", "roll_std_28"]).copy()
X = pd.get_dummies(modele_df[features_num + features_cat], columns=features_cat, drop_first=True)
y = modele_df["units_sold"]

coupure = modele_df["date"].max() - pd.Timedelta(days=JOURS_TEST)
train, test = (modele_df["date"] <= coupure), (modele_df["date"] > coupure)
print(f"Entraînement : {train.sum():,} lignes | Test : {test.sum():,} lignes (après {coupure.date()})")

def scores(y_vrai, y_pred):
    err = y_vrai - y_pred
    return {"MAE": err.abs().mean(), "RMSE": np.sqrt((err ** 2).mean()),
            "WAPE_%": 100 * err.abs().sum() / y_vrai.sum(), "Biais": err.mean()}

resultats = {}
resultats["Référence : même jour S-1 (lag_7)"] = scores(y[test], modele_df.loc[test, "lag_7"])
resultats["Référence : moyenne glissante 7 j"] = scores(y[test], modele_df.loc[test, "roll_mean_7"])
if "demand_forecast" in modele_df.columns:
    resultats["Prévision fournie (demand_forecast)"] = scores(y[test], modele_df.loc[test, "demand_forecast"])

gbm = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=6, random_state=SEED)
gbm.fit(X[train], y[train])
pred = pd.Series(gbm.predict(X[test]).clip(min=0), index=y[test].index)
resultats["Modèle : Gradient Boosting"] = scores(y[test], pred)

comparaison = pd.DataFrame(resultats).T.sort_values("WAPE_%")
comparaison

In [ ]:
# Prévision agrégée sur la période de test
agr = pd.DataFrame({"reel": y[test], "prevu": pred, "date": modele_df.loc[test, "date"]}).groupby("date").sum()
agr.plot(lw=1.5, color=["#333333", "#F2A900"])
plt.title("Période de test : ventes réelles et prévues (total par jour)"); plt.ylabel("Unités"); plt.show()

# Variables les plus utiles (importance par permutation, sur un échantillon du test)
idx = X[test].sample(min(4000, int(test.sum())), random_state=SEED).index
imp = permutation_importance(gbm, X.loc[idx], y.loc[idx], n_repeats=3, random_state=SEED)
top = pd.Series(imp.importances_mean, index=X.columns).sort_values().tail(10)
plt.barh(list(top.index), top.values, color="#4472C4")
plt.title("Importance des variables (permutation)"); plt.xlabel("Baisse de score si la variable est mélangée")
plt.tight_layout(); plt.show()

### 8.2 Recommandation de quantité à commander
La demande moyenne prévue sur les 7 derniers jours de test remplace la moyenne historique, puis on applique la même politique de stock (stock de sécurité, niveau cible).

In [ ]:
derniers_7 = modele_df.loc[test, "date"] > modele_df["date"].max() - pd.Timedelta(days=7)
mu_prevu = (pd.DataFrame({"store_id": modele_df.loc[test, "store_id"], "product_id": modele_df.loc[test, "product_id"],
                          "prevu": pred})[derniers_7]
            .groupby(grp)["prevu"].mean().rename("demande_prevue").reset_index())

reco = kpi[grp + ["stock_actuel", "demande_moy", "demande_std", "classe_ABC"]].merge(mu_prevu, on=grp, how="left")
reco["demande_prevue"] = reco["demande_prevue"].fillna(reco["demande_moy"])  # repli : moyenne historique
reco["stock_securite"] = Z_SERVICE * reco["demande_std"] * np.sqrt(L)
reco["point_commande"] = reco["demande_prevue"] * L + reco["stock_securite"]
reco["niveau_cible"] = reco["demande_prevue"] * (L + R) + reco["stock_securite"]
reco["qte_recommandee"] = np.ceil((reco["niveau_cible"] - reco["stock_actuel"]).clip(lower=0))
reco["statut"] = np.where(reco["stock_actuel"] < reco["point_commande"], "Risque de rupture", "OK")

print("Séries à réapprovisionner :", int((reco["qte_recommandee"] > 0).sum()), "sur", len(reco))
reco.sort_values("qte_recommandee", ascending=False).head(15)

## 9. Export des résultats

In [ ]:
df.to_csv(OUTPUT_DIR / "donnees_nettoyees.csv", index=False)
kpi.to_csv(OUTPUT_DIR / "kpi_series.csv", index=False)
reco.to_csv(OUTPUT_DIR / "recommandations_commande.csv", index=False)
anomalies.to_csv(OUTPUT_DIR / "anomalies.csv", index=False)
comparaison.to_csv(OUTPUT_DIR / "comparaison_modeles.csv")
pd.DataFrame(rapports).to_csv(OUTPUT_DIR / "rapport_qualite.csv", index=False)
(OUTPUT_DIR / "journal_nettoyage.txt").write_text("\n".join(journal), encoding="utf-8")

print("Fichiers exportés dans", OUTPUT_DIR.resolve())
for f in sorted(OUTPUT_DIR.iterdir()):
    print(" -", f.name)

## Conclusion et pistes

- **Qualité** : voir le rapport avant / après et le journal de nettoyage.
- **Demande** : tendance, saisonnalité (mois, jour de semaine) et effet des promotions mesurés aux sections 4 et 5.
- **Modèle** : comparer le gradient boosting aux références simples (section 8.1). Un modèle n'est utile que s'il fait mieux que la moyenne glissante.
- **Stock** : séries à risque de rupture et quantités recommandées (sections 7 et 8.2).
- **Limites** : le délai de livraison (`LEAD_TIME_JOURS`) est une hypothèse. Le dataset Kaggle est synthétique : les corrélations réelles peuvent être faibles. Pour Marjane, remplacer par les données réelles (ERP, WMS, POS).
- **Prochaines étapes** : modèles par famille de produits, intégration des vrais délais fournisseurs, automatisation du traitement (workflow Zapier), tableau de bord.